# [7] AttentiveSentinel Model suggested by [@gtend](https://github.com/gtend) (유채민)

In [207]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import re
import numpy as np
from transformers import AutoTokenizer, AutoModel

# 3. 환경 설정 및 문장 분리 함수
print("\n환경 설정 중...")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def split_sentences_without_library(text):
    """설치 없는 문장 분리기"""
    sentences = re.split(r'(?<=[.?!])\s+', text)
    return [s.strip() for s in sentences if s]

# 4. 문장별 어텐션 모델 정의 (수정된 부분)
class SentenceAttentionModel(nn.Module):
    def __init__(self, model_name='klue/bert-base', num_classes=2):
        super(SentenceAttentionModel, self).__init__()
        print(f"'{model_name}' 모델 로드 중...")
        # 1. 문장 내 토큰 임베딩을 위한 BERT 모델
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embedding_dim = self.bert.config.hidden_size
        
        # 2. 어텐션 가중치를 계산하기 위한 신경망
        self.attention_net = nn.Sequential(
            nn.Linear(self.embedding_dim, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )
        
        # 3. 최종 분류를 위한 분류기
        self.classifier = nn.Linear(self.embedding_dim, num_classes)
        print("모델 생성 완료!")

    # 평균 풀링을 수행하는 헬퍼 함수
    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, sentences: list):
        # 1. 문장 리스트 토큰화
        tokenized_inputs = self.tokenizer(
            sentences,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(DEVICE)

        # 2. BERT 모델 통과
        with torch.no_grad():
            bert_output = self.bert(**tokenized_inputs)

        # 3. 평균 풀링으로 문장 임베딩 생성
        sentence_embeddings = self._mean_pooling(bert_output, tokenized_inputs['attention_mask'])
        
        # 4. 어텐션 점수 계산
        attention_raw_scores = self.attention_net(sentence_embeddings)
        attention_weights = F.softmax(attention_raw_scores, dim=0)
        
        # 5. 가중 평균으로 문서 벡터 생성
        document_vector = torch.sum(sentence_embeddings * attention_weights, dim=0)
        
        # 6. 최종 분류
        logits = self.classifier(document_vector)
        
        return logits, attention_weights


# 5. 파일럿 테스트 실행 (이전과 동일)
print("\n파일럿 테스트 시작...")




# print(X["full_text"][i])

model = SentenceAttentionModel().to(DEVICE)
model.eval()

# sample_text = "이 프로젝트는 성공적으로 마무리되었습니다. 하지만 몇 가지 아쉬운 점이 남았습니다. 특히 예산 문제가 가장 컸습니다. 다음에는 더 나은 결과를 기대합니다."
i = 20
sample_text = X["full_text"][i]

sentences = split_sentences_without_library(sample_text)

with torch.no_grad():
    final_logits, attention_scores = model(sentences)

final_prediction = torch.argmax(final_logits).cpu().item()
attention_scores = attention_scores.cpu().numpy().flatten()

print("-" * 30)
print("--- 최종 결과 ---")
print(f"최종 예측 결과 (0: Human, 1: AI): {final_prediction}")
print("\n[문장별 어텐션 점수 분석]")

def make_bar(score, max_len=20):
    return '█' * int(score * max_len)

for sentence, score in zip(sentences, attention_scores):
    print(y[i])
    print(f"점수: {score:.4f} | {make_bar(score)} | 문장: {sentence}")


환경 설정 중...

파일럿 테스트 시작...
'klue/bert-base' 모델 로드 중...
모델 생성 완료!
------------------------------
--- 최종 결과 ---
최종 예측 결과 (0: Human, 1: AI): 1

[문장별 어텐션 점수 분석]
1
점수: 0.0524 | █ | 문장: 수난곡(受難曲)은 배우의 연기 없이 무대에 올려지는 성악을 주로 한 종합 예술이다.
1
점수: 0.0452 |  | 문장: 이러한 의미에서 오라토리오와 유사하지만, 성경의 사복음서를 기반으로 한 예수 그리스도의 생애를 주로 다루고 있다는 점에서 차이가 있다.
1
점수: 0.0626 | █ | 문장: 또한 이는 주로 독일 계열 작곡가들에게 쓰인 개념이다.
1
점수: 0.0575 | █ | 문장: 수난 또는 수난곡을 뜻하는 영어 'Passion'은 2세기에 나타난 라틴어 "passio"에서 유래하며, 예수의 생애와 고난이란 의미를 담고 있다.
1
점수: 0.0672 | █ | 문장: 사복음서에 기록된 예수님의 수난 이야기는 마태오, 마르코, 루카, 요한 복음서에 각각 담겨 있습니다.
1
점수: 0.0606 | █ | 문장: 이를 바탕으로 '마태오 수난곡', '마르코 수난곡', '루카 수난곡', '요한 수난곡'이라는 4개의 작품이 만들어졌죠.
1
점수: 0.0599 | █ | 문장: 이들 작품은 예수님의 고난과 십자가 처형을 다루고 있습니다.
1
점수: 0.0609 | █ | 문장: 옛날부터 성(聖) 금요일이나 성주간(聖週間)에는 수난극이나 이와 비슷한 행사를 하였다.
1
점수: 0.0530 | █ | 문장: 12세기경부터 복음서에 따라 그리스도 수난의 이야기를 3인의 신부가, 한 사람은 복음사가(福音史家)의 역(테너)을, 또 한 사람은 그리스도의 역(베이스)을, 나머지 한 사람은 군중의 역(알토)을 맡아 낭독조로 노래하는 습관이 되었다.
1
점수: 0.0648 | █ | 문장: 이것이 그 뒤의 수난곡의 기원이다.
1
점수: 0.0673 | █ | 

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel, AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
from sklearn.utils import resample


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# MODEL_NAME = "monologg/koelectra-small-v3-discriminator" 
MODEL_NAME = "monologg/koelectra-base-v3-discriminator" 
NUM_EPOCHS = 30       # 테스트를 위해 에폭 수를 작게 설정 (실제로는 3~5)
BATCH_SIZE = 4       # GPU 메모리에 따라 조정 (V100/A100 기준 8~16 가능)
# LEARNING_RATE = 2e-4 # BERT Fine-tuning에 일반적으로 사용되는 학습률
LEARNING_RATE = 2e-5

train = pd.read_csv("data/train.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")# 2. 라벨별 분리
label1 = train[train["generated"] == 1]
label0 = train[train["generated"] == 0]

# 3. 0인 데이터 수를 1인 데이터 수만큼만 샘플링
label0_balanced = resample(label0,
                           replace=False,
                           n_samples=len(label1)*2,
                           random_state=42)

# 4. 합치기 및 셔플
train_balanced = pd.concat([label1, label0_balanced])
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)



X = train_balanced['full_text']
y = train_balanced['generated']
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

print("데이터 준비 완료.")

데이터 준비 완료.


In [4]:
from torch.cuda.amp import GradScaler, autocast


# --- PyTorch Dataset 정의 ---
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        return {'text': text, 'label': torch.tensor(label)}

# --- 문장 분리기 ---
def split_sentences_without_library(text):

    sentences = re.split(r'(?<=[.?!])\s+', text)
    return [s.strip() for s in sentences if s]

# --- DataLoader를 위한 Collate 함수 ---
# 배치 내에서 문장 리스트를 처리하기 위한 함수
def collate_fn(batch):
    texts = [item['text'] for item in batch]
    labels = [item['label'] for item in batch]
    return {'texts': texts, 'labels': torch.stack(labels)}

train_dataset = TextDataset(X_train.values, y_train.values)
val_dataset = TextDataset(X_val.values, y_val.values)


train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


In [5]:
# ==============================================================================
# 5. 모델 정의 (Model Definition)
# ==============================================================================
class SentenceAttentionModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_classes=2):
        super(SentenceAttentionModel, self).__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.embedding_dim = self.bert.config.hidden_size
        self.attention_net = nn.Sequential(nn.Linear(self.embedding_dim, 256), nn.Tanh(), nn.Linear(256, 1))
        self.dropout_rate = 0.3
        self.dropout = nn.Dropout(self.dropout_rate)
        # self.classifier = nn.Linear(self.embedding_dim, num_classes)
        
        self.hidden_size = 256 # 중간 은닉층의 크기

        self.classifier = nn.Sequential(
            nn.Linear(self.embedding_dim, self.hidden_size), # 첫 번째 레이어: 문서벡터 -> 은닉층
            nn.ReLU(),                                  # 비선형성을 더해줄 활성화 함수
            nn.Dropout(self.dropout_rate),                   # 과적합 방지를 위한 드롭아웃
            nn.Linear(self.hidden_size, num_classes)         # 두 번째 레이어: 은닉층 -> 최종 출력
        )
        
    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def forward(self, texts: list):
        batch_logits = []
        for text in texts:
            sentences = split_sentences_without_library(text)
            if not sentences: continue
            
            tokenized = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt').to(DEVICE)
            
            # BERT 통과 시에는 그래디언트 계산을 하지 않음 (메모리 및 속도 향상)
            with torch.no_grad():
                bert_output = self.bert(**tokenized)
            
            sentence_embeddings = self._mean_pooling(bert_output, tokenized['attention_mask'])
            
            attention_scores = F.softmax(self.attention_net(sentence_embeddings), dim=0)
            document_vector = torch.sum(sentence_embeddings * attention_scores, dim=0)
            
            # document_vector_with_dropout = self.dropout(document_vector)
            # logits = self.classifier(document_vector)
            
            logits = self.classifier(document_vector)
            batch_logits.append(logits)
        
        return torch.stack(batch_logits) if batch_logits else None

In [6]:
# ==============================================================================
# 6. 학습 준비 (Training Setup)
# ==============================================================================
print("\n학습 준비 중...")
model = SentenceAttentionModel().to(DEVICE)

# --- BERT 파라미터 동결 ---
print("사전 학습된 BERT 모델의 파라미터를 동결합니다...")
for param in model.bert.parameters():
    param.requires_grad = False

# num_layers_to_unfreeze = 2 
# print(f"KoELECTRA의 마지막 {num_layers_to_unfreeze}개 레이어의 동결을 해제합니다...")
# for layer in model.bert.encoder.layer[-num_layers_to_unfreeze:]:
#     for param in layer.parameters():
#         param.requires_grad = True
        
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = AdamW(trainable_params, lr=LEARNING_RATE) # lr=5e-5 등 fine-tuning에 맞는 값 추천

# 학습이 필요한 파라미터만 옵티마이저에 전달
# trainable_params = filter(lambda p: p.requires_grad, model.parameters())
# optimizer = AdamW(trainable_params, lr=LEARNING_RATE)

#class_weights = torch.tensor([1.0, 3.0], device=device)
#criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("학습 준비 완료.")

# ==============================================================================
# 7. 학습 및 검증 루프 (Training & Validation Loop)
# ==============================================================================
best_f1 = 0
print("\n학습 시작!")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    all_train_preds, all_train_labels = [], []
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for batch in progress_bar:
        optimizer.zero_grad()
        
        texts = batch['texts']
        labels = batch['labels'].to(DEVICE)
        
        logits = model(texts)
        if logits is None: continue
        
        loss = criterion(logits, labels)
        
        total_train_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        
        preds = torch.argmax(logits, dim=1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'Train Loss': loss.item()})
        
    avg_train_loss = total_train_loss / len(train_dataloader)
    train_f1 = f1_score(all_train_labels, all_train_preds, average='macro')

    # --- 검증 모드 ---
    model.eval()
    total_val_loss = 0
    all_val_preds, all_val_labels = [], []

    with torch.no_grad():
        progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]")
        for batch in progress_bar:
            texts = batch['texts']
            labels = batch['labels'].to(DEVICE)

            logits = model(texts)
            if logits is None: continue
            
            loss = criterion(logits, labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')
    
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Train F1: {train_f1:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1:.4f}")
    print("-" * 80)
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "attmodelV6.pt")
        print("✅ 모델 저장 완료 (AUC 갱신)")


학습 준비 중...


2025-07-04 10:11:58.972824: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-04 10:11:59.120963: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


사전 학습된 BERT 모델의 파라미터를 동결합니다...


/usr/local/lib/python3.8/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


학습 준비 완료.

학습 시작!


...

Epoch 5/30 [Train]:   0%|          | 0/6396 [00:00<?, ?it/s]

Epoch 5/30 [Val]:   0%|          | 0/1599 [00:00<?, ?it/s]


Epoch 5/30 | Train Loss: 0.4992 | Train F1: 0.8299 | Val Loss: 0.5302 | Val F1: 0.7422
--------------------------------------------------------------------------------


Epoch 6/30 [Train]:   0%|          | 0/6396 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# torch.save(model.state_dict(), "attmodelV5.pt")
# print("✅ 모델 저장 완료 (AUC 갱신)")

In [7]:
import matplotlib.pyplot as plt

In [8]:
# logits = torch.tensor([[0.4728, 0.5272],
#                        [0.3384, 0.6616],
#                        [0.7813, 0.2187],
#                        [0.9810, 0.0190]])

In [9]:
# p_ai = logits[:, 1]
# p_ai

In [10]:
# probs = F.softmax(logits, dim=1)
# print(probs)
# max_probs = torch.max(probs, dim=1).values
# max_probs

In [11]:
# probs = probs[0]
# if probs[0] > probs[1]:
#     final = probs[1]
# else :
#     final = probs[0]
# print(probs)
# print(final)

In [ ]:
from collections import Counter

# --- 검증 모드 ---
model.eval()
total_val_loss = 0
all_val_preds, all_val_probs, all_val_labels = [], [], []

with torch.no_grad():
    progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]")
    for batch in progress_bar:
        texts = batch['texts']
        labels = batch['labels'].to(DEVICE)

        logits = model(texts)
        if logits is None: continue

        # loss = criterion(logits, labels)
        # total_val_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        # preds = F.softmax(logits, dim=1)

        
        probs = F.softmax(logits, dim=1)
        # print(probs)
        # max_probs = torch.max(probs, dim=1).values
        probs = probs[:, 1]
        
        # print("max : ", probs)
        # probs = probs[0]
        # if probs[0] > probs[1]:
        #     final = probs[1]
        # else :
        #     final = probs[0]


        # index = torch.argmax(preds, dim=1)
        
        # probs = preds[index]
        
        # print(probs)
        # print(logits.shape)
        # print(preds)
        all_val_preds.extend(preds.cpu().numpy())        
        all_val_probs.extend(probs.cpu().numpy())
        all_val_labels.extend(labels.cpu().numpy())

Epoch 6/30 [Val]:   0%|          | 0/1599 [00:00<?, ?it/s]

In [ ]:
all_val_probs

In [ ]:
#avg_val_loss = total_val_loss / len(val_dataloader)
# val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')

# 2) 라벨 분포 계산
label_counts = Counter(all_val_labels)

# 3) 라벨별 확률 분리
probs_by_label = {0: [], 1: []}
for lbl, p in zip(all_val_labels, all_val_preds):
    probs_by_label[lbl].append(p)

# 4) 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (a) 라벨 분포
axes[0].bar(label_counts.keys(), label_counts.values(), width=0.4, edgecolor='black')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Label 0', 'Label 1'])
axes[0].set_title('검증셋 라벨 분포')
axes[0].set_xlabel('라벨')
axes[0].set_ylabel('샘플 수')

# (b) 라벨별 예측 확률 분포
bins = 50
axes[1].hist([probs_by_label[0], probs_by_label[1]],
             bins=bins,
             label=['Label 0', 'Label 1'],
             alpha=0.7,
             edgecolor='black')
axes[1].set_title('라벨별 예측 확률 분포')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('샘플 수')
axes[1].legend()
from collections import Counter


plt.tight_layout()
plt.show()

In [ ]:
#avg_val_loss = total_val_loss / len(val_dataloader)
# val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')

# 2) 라벨 분포 계산
label_counts = Counter(all_val_labels)

# 3) 라벨별 확률 분리
probs_by_label = {0: [], 1: []}
for lbl, p in zip(all_train_labels, all_train_preds):
    probs_by_label[lbl].append(p)

# 4) 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (a) 라벨 분포
axes[0].bar(label_counts.keys(), label_counts.values(), width=0.4, edgecolor='black')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Label 0', 'Label 1'])
axes[0].set_title('검증셋 라벨 분포')
axes[0].set_xlabel('라벨')
axes[0].set_ylabel('샘플 수')

# (b) 라벨별 예측 확률 분포
bins = 50
axes[1].hist([probs_by_label[0], probs_by_label[1]],
             bins=bins,
             label=['Label 0', 'Label 1'],
             alpha=0.7,
             edgecolor='black')
axes[1].set_title('라벨별 예측 확률 분포')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('샘플 수')
axes[1].legend()
from collections import Counter


plt.tight_layout()
plt.show()

In [ ]:
#avg_val_loss = total_val_loss / len(val_dataloader)
# val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')

# 2) 라벨 분포 계산
label_counts = Counter(all_val_labels)

# 3) 라벨별 확률 분리
probs_by_label = {0: [], 1: []}
for lbl, p in zip(all_val_labels, all_val_probs):
    probs_by_label[lbl].append(p)

# 4) 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (a) 라벨 분포
axes[0].bar(label_counts.keys(), label_counts.values(), width=0.4, edgecolor='black')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Label 0', 'Label 1'])
axes[0].set_title('검증셋 라벨 분포')
axes[0].set_xlabel('라벨')
axes[0].set_ylabel('샘플 수')

# (b) 라벨별 예측 확률 분포
bins = 50
axes[1].hist([probs_by_label[0], probs_by_label[1]],
             bins=bins,
             label=['Label 0', 'Label 1'],
             alpha=0.7,
             edgecolor='black')
axes[1].set_title('라벨별 예측 확률 분포')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('샘플 수')
axes[1].legend()
from collections import Counter


plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_train_labels, all_train_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real(0)", "Generated(1)"])

# confusion matrix 출력
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_val_labels, all_val_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real(0)", "Generated(1)"])

# confusion matrix 출력
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader

# --- 데이터셋, 문장 분리기, 콜레이트 함수 (학습 시 사용했던 것과 동일) ---
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {'text': self.texts[idx], 'label': torch.tensor(self.labels[idx])}

def split_sentences_without_library(text):
    if not isinstance(text, str): return []
    return [s.strip() for s in re.split(r'(?<=[.?!])\s+', text) if s]

def collate_fn(batch):
    return {'texts': [item['text'] for item in batch], 
            'labels': torch.stack([item['label'] for item in batch])}


# ==============================================================================
# 1. test 데이터 로드 및 전처리
# ==============================================================================
print("테스트 데이터를 준비합니다...")
# ⚠️ 파일 경로를 실제 환경에 맞게 수정해주세요.
test = pd.read_csv('./data/test.csv', encoding='utf-8-sig')
test = test.rename(columns={'paragraph_text': 'full_text'})

# title과 full_text를 하나의 텍스트로 결합
X_test_combined = test['title'].fillna('') + " " + test['full_text'].fillna('')

# test 데이터에는 라벨이 없으므로, 더미(dummy) 라벨 생성
dummy_labels = np.zeros(len(X_test_combined))

# ==============================================================================
# 2. Test Dataloader 생성
# ==============================================================================
# 수정된 데이터로 TextDataset 및 DataLoader 생성
test_dataset = TextDataset(X_test_combined.values, dummy_labels)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

print("Test DataLoader가 성공적으로 생성되었습니다.")

In [ ]:
print(len(test_dataset))
print(len(test_dataloader))

In [ ]:
from collections import Counter

# --- Test 모드 ---
model.eval()
total_val_loss = 0
all_test_preds, all_test_labels = [], []

with torch.no_grad():
    progress_bar = tqdm(test_dataloader, desc="Predicting")
    for batch in progress_bar:
        texts = batch['texts']
        labels = batch['labels'].to(DEVICE)
        logits = model(texts)
        # print(text)

        if logits is None: continue

        # preds = F.softmax(logits, dim=1)
    
        preds = torch.argmax(logits, dim=1)
        
        probs = F.softmax(logits, dim=1)
        # print(probs)
        # max_probs = torch.max(probs, dim=1).values
        probs = probs[:, 1]
        # index = torch.argmax(preds, dim=1)
        
        # probs = preds[index]
        
        # print(probs)
        # print(logits.shape)
        # print(preds)
        all_test_preds.extend(probs.cpu().numpy())
        
sample_submission = pd.read_csv('./data/sample_submission.csv', encoding='utf-8-sig')
sample_submission['generated'] = all_test_preds

sample_submission.to_csv(f'./att_submissionV6_probs.csv', index=False)



print("✅ 예측 결과 저장 완료 → att_submissionV6_probs.csv")

In [ ]:
logits

In [ ]:
# probs = F.softmax(logits, dim=1)
# print(probs)
# probs = probs[:, 1]
# print(probs)

In [ ]:
from collections import Counter

# --- Test 모드 ---
model.eval()
total_val_loss = 0
all_test_preds, all_test_labels = [], []

with torch.no_grad():
    progress_bar = tqdm(test_dataloader, desc="Predicting")
    for batch in progress_bar:
        texts = batch['texts']
        # labels = batch['labels'].to(DEVICE) # test 데이터에는 라벨이 없습니다.
        
        logits = model(texts)
        if logits is None: continue

        # 1. AI(클래스 1)일 확률을 계산합니다.
        probs_tensor = F.softmax(logits, dim=1)[:, 1]
        probs_np = probs_tensor.cpu().numpy()

        # 2. 커스텀 임계값 적용
        # 0.6 이상이면 1로, 0.4 이하면 0으로, 그 사이는 원래 확률값으로 유지
        # 실제 제출 시에는 0.4~0.6 사이 값도 0 또는 1로 결정해야 할 수 있습니다.
        thresholded_preds = np.where(probs_np >= 0.7, 1, 
                                     np.where(probs_np <= 0.3, 0, probs_np))
        
        all_test_preds.extend(thresholded_preds)
        
sample_submission = pd.read_csv('./data/sample_submission.csv', encoding='utf-8-sig')
sample_submission['generated'] = all_test_preds

sample_submission.to_csv(f'./att_submissionV6.csv', index=False)



print("✅ 예측 결과 저장 완료 → att_submissionV6.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트가 깨지지 않도록 설정 (환경에 따라 필요한 경우)
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False

# 1. CSV 파일 로드
try:
    df = pd.read_csv('att_submissionV6.csv')
    print("파일 로드 완료.")
except FileNotFoundError:
    print("오류: 'att_submission.csv' 파일을 찾을 수 없습니다.")
    exit()

# 2. 시각화
plt.figure(figsize=(12, 7)) # 그래프 크기 설정

# seaborn의 histplot을 사용하여 히스토그램과 KDE 곡선을 함께 그림
sns.histplot(data=df, x='generated', kde=True, bins=50)

# 3. 그래프 제목 및 라벨 설정
plt.title('Prediction Probability Distribution', fontsize=16)
plt.xlabel('Predicted Probability (Generated = 1)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

# 4. 그래프 표시
plt.show()

# 5. 기초 통계량 출력 (참고용)
print("\n[기초 통계량]")
print(df['generated'].describe())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트가 깨지지 않도록 설정 (환경에 따라 필요한 경우)
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False

# 1. CSV 파일 로드
try:
    df = pd.read_csv('att_submissionV6_probs.csv')
    print("파일 로드 완료.")
except FileNotFoundError:
    print("오류: 'att_submission.csv' 파일을 찾을 수 없습니다.")
    exit()

# 2. 시각화
plt.figure(figsize=(12, 7)) # 그래프 크기 설정

# seaborn의 histplot을 사용하여 히스토그램과 KDE 곡선을 함께 그림
sns.histplot(data=df, x='generated', kde=True, bins=50)

# 3. 그래프 제목 및 라벨 설정
plt.title('Prediction Probability Distribution (ensemble3.csv)', fontsize=16)
plt.xlabel('Predicted Probability (Generated = 1)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

# 4. 그래프 표시
plt.show()

# 5. 기초 통계량 출력 (참고용)
print("\n[기초 통계량]")
print(df['generated'].describe())

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
overall_auc = roc_auc_score(all_train_labels, all_train_preds)
print(f"\n🔍 Overall Validation AUC (from probabilities): {overall_auc:.4f}")

In [ ]:
overall_auc = roc_auc_score(all_val_labels, all_val_preds)
print(f"\n🔍 Overall Validation AUC (from probabilities): {overall_auc:.4f}")

In [ ]:
overall_auc = roc_auc_score(all_val_labels, all_val_probs)
print(f"\n🔍 Overall Validation AUC (from probabilities): {overall_auc:.4f}")

In [90]:
X_test_combined

0       공중 도덕의 의의와 필요성 도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동...
1       공중 도덕의 의의와 필요성 도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없...
2       공중 도덕의 의의와 필요성 여기에 이른바 공중도덕은 실천적, 사회적 도덕의 한 부문...
3       공중 도덕의 의의와 필요성 우리가 공동 생활을 하는 데 있어서 공중 도덕이 필요함은...
4       풍습과 그 개선 인간 사회에서는 다 함께 지켜야 할 어떤 기준이 있어 이를 따르면 ...
                              ...                        
1957    저작권! 내가 먼저 지켜야지 인터넷에는 음악뿐만 아니라 인터넷소설, 영화, 애니메이...
1958    저작권! 내가 먼저 지켜야지 하지만 이 경험을 통해 나는 달라진 시각을 갖게 되었다...
1959    저작권! 내가 먼저 지켜야지 그런데 누군가는 아무 노력이나 허락 없이 사용한다면 불...
1960    저작권! 내가 먼저 지켜야지 우리들이 우리나라의 노래, 영화, 드라마, 애니메이션 ...
1961    저작권! 내가 먼저 지켜야지 우리나라의 저작권법! 우리들부터 정확히 알고 지켜나갈 ...
Length: 1962, dtype: object

In [91]:
# model.load_state_dict(torch.load("attmodelV5.pt"))

In [94]:
paragraphs['full_text']

0       도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다. 그러므로 도덕...
1       도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없다. 개인적인 측면과 사회적...
2       여기에 이른바 공중도덕은 실천적, 사회적 도덕의 한 부문이다. 즉, 공중 도덕이라 ...
3       우리가 공동 생활을 하는 데 있어서 공중 도덕이 필요함은 위에서 말한 것처럼 알 수...
4       인간 사회에서는 다 함께 지켜야 할 어떤 기준이 있어 이를 따르면 옳다고 하고 따르...
                              ...                        
1957    인터넷에는 음악뿐만 아니라 인터넷소설, 영화, 애니메이션 등 내가 좋아하는 것들이 ...
1958    하지만 이 경험을 통해 나는 달라진 시각을 갖게 되었다. 이제는 내가 좋아하는 콘텐...
1959    그런데 누군가는 아무 노력이나 허락 없이 사용한다면 불공평하다. 우리나라는 인기 있...
1960    우리들이 우리나라의 노래, 영화, 드라마, 애니메이션 등의 저작권을 지켜 주어야 다...
1961    우리나라의 저작권법! 우리들부터 정확히 알고 지켜나갈 때, 더욱 더 우리나라는 발전...
Name: full_text, Length: 1962, dtype: object

In [100]:
paragraphs = test[['title', 'full_text']].copy()
paragraphs["title"]

0        공중 도덕의 의의와 필요성
1        공중 도덕의 의의와 필요성
2        공중 도덕의 의의와 필요성
3        공중 도덕의 의의와 필요성
4              풍습과 그 개선
             ...       
1957    저작권! 내가 먼저 지켜야지
1958    저작권! 내가 먼저 지켜야지
1959    저작권! 내가 먼저 지켜야지
1960    저작권! 내가 먼저 지켜야지
1961    저작권! 내가 먼저 지켜야지
Name: title, Length: 1962, dtype: object

In [103]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from tqdm import tqdm
import re

# 가정: model은 이미 학습된 상태로 로드되어 있음
# from model_definition import SentenceAttentionModel
# model = SentenceAttentionModel().to(DEVICE)
# model.load_state_dict(torch.load("best_model.pt"))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Step 1: test.csv 불러오기
test = pd.read_csv('./data/test.csv', encoding='utf-8-sig')
test = test.rename(columns={'paragraph_text': 'full_text'})

# Step 2: title별로 문단 병합 (문단을 이어붙인 전체 글)
grouped_texts = test.groupby('title')['full_text'].apply(lambda x: ' '.join(x)).reset_index()
title_to_fulltext = dict(zip(grouped_texts['title'], grouped_texts['full_text']))

# Step 3: 원래 문단 (개별 문단 단위)도 따로 유지
paragraphs = test[['title', 'full_text']].copy()


# Step 4: 문단에 해당하는 title 기준 전체 글을 병합하여 column 추가
paragraphs['combined_text'] = paragraphs['title'].map(title_to_fulltext)
paragraphs['merged_input'] = paragraphs['title'].fillna('') + " " + paragraphs['full_text'].fillna('')

# Step 5: 더미 라벨
dummy_labels = np.zeros(len(paragraphs))

# Step 6: 새로운 Dataset
class ParagraphDataset(Dataset):
    def __init__(self, full_texts, combined_texts, labels):
        self.full_texts = full_texts
        self.combined_texts = combined_texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'paragraph': self.full_texts[idx],
            'full_doc': self.combined_texts[idx],
            'label': torch.tensor(self.labels[idx])
        }

def collate_fn(batch):
    return {
        'paragraphs': [item['paragraph'] for item in batch],
        'docs': [item['full_doc'] for item in batch],
        'labels': torch.stack([item['label'] for item in batch])
    }

# Step 7: DataLoader
test_dataset = ParagraphDataset(
    full_texts=paragraphs['merged_input'].values,
    combined_texts=paragraphs['combined_text'].values,
    labels=dummy_labels
)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# Step 8: 추론
model.eval()
doc_preds = []
para_preds = []

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Predicting"):
        # 문단 단위 예측
        para_logits = model(batch['paragraphs'])
        para_probs = F.softmax(para_logits, dim=1)[:, 1]

        # 문서 전체 단위 예측
        doc_logits = model(batch['docs'])
        doc_probs = F.softmax(doc_logits, dim=1)[:, 1]

        para_preds.extend(para_probs.cpu().numpy())
        doc_preds.extend(doc_probs.cpu().numpy())

# Step 9: 결과 저장
output = pd.DataFrame({
    'title': paragraphs['title'],
    'paragraph': paragraphs['full_text'],
    'doc_prob': doc_preds,
    'para_prob': para_preds
})

Predicting: 100% 1962/1962 [02:07<00:00, 15.34it/s]


In [107]:
output[:10]

,title,paragraph,doc_prob,para_prob
0,공중 도덕의 의의와 필요성,도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다. 그러므로 도덕...,0.673039,0.633318
1,공중 도덕의 의의와 필요성,도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없다. 개인적인 측면과 사회적...,0.673039,0.262233
2,공중 도덕의 의의와 필요성,"여기에 이른바 공중도덕은 실천적, 사회적 도덕의 한 부문이다. 즉, 공중 도덕이라 ...",0.673039,0.127238
3,공중 도덕의 의의와 필요성,우리가 공동 생활을 하는 데 있어서 공중 도덕이 필요함은 위에서 말한 것처럼 알 수...,0.673039,0.903172
4,풍습과 그 개선,인간 사회에서는 다 함께 지켜야 할 어떤 기준이 있어 이를 따르면 옳다고 하고 따르...,0.958498,0.909643
5,풍습과 그 개선,"풍습은 도덕을 의미하고 법률의 성질을 가진 것이었으나, 인지가 발달함에 따라 도덕과...",0.958498,0.980695
6,풍습과 그 개선,"예의는 기원이 오래되고, 농업을 유일한 생업으로 삼고 봉건적 대가족 제도 아래에서 ...",0.958498,0.970109
7,생활의 안정과 가정,행복한 가정을 이루자면 어느 정도의 경제적 안정이 있을 필요가 있다. 직접에 충실하...,0.547297,0.796050
8,생활의 안정과 가정,그런 경제적 안정을 얻는다고 해서 그것이 곧 행복한 가정을 의미하는 것은 아니라는 ...,0.547297,0.647357
9,생활의 안정과 가정,나아가 우리가 알아두어야 할 것은 가정 생활이란 자기 한 가정만으로써 행복을 누리는...,0.547297,0.093649


In [116]:
output[1700:1710]

,title,paragraph,doc_prob,para_prob
1700,특명! 뭉이와 뭉글이를 보호하라!,나는 그림을 그리고 디자인하고 만드는 것을 좋아한다. 그러기에 내 디자인노트에는 내...,0.9741,0.570101
1701,특명! 뭉이와 뭉글이를 보호하라!,새하얀 도화지 위에 많은 사람들에게 소개될 준비를 마친 뭉이와 뭉글이가 있다. 이들...,0.9741,0.746431
1702,특명! 뭉이와 뭉글이를 보호하라!,‘양심 나라’에 도착한 뭉이는 제일 먼저 어느 멋진 회사에 자기 이름이 등록된다. ...,0.9741,0.999987
1703,특명! 뭉이와 뭉글이를 보호하라!,"그러고서 이 아주머니는 뭉이를 사용할 수 있도록 적당한 금액을 지불하고, 자신이 디...",0.9741,0.954643
1704,특명! 뭉이와 뭉글이를 보호하라!,"“뭉이야, 축하해~ 네가 들어간 여러 상품들이 사람들에게 많은 사랑을 받으면서 디자...",0.9741,0.941767
1705,특명! 뭉이와 뭉글이를 보호하라!,"한편, ‘나 몰라 나라’에 도착한 뭉글이는 제일 먼저 어느 으스스한 공장에 도착한다...",0.9741,0.092853
1706,특명! 뭉이와 뭉글이를 보호하라!,"이때, 어느 한 아저씨가 뭉글이의 이름을 묻는다. ""이 캐릭터 이름은 뭐지? 어디서...",0.9741,0.893418
1707,특명! 뭉이와 뭉글이를 보호하라!,"“그럼, 이거 사가지고 온 거야?” 한 아저씨가 묻자, “사가지고 오긴…. 뭘 사가...",0.9741,0.999634
1708,특명! 뭉이와 뭉글이를 보호하라!,뭉글이는 옆에서 자신과 비슷하기도 하고 어딘가 엉성한 친구들을 발견한다. 뭉글이는 ...,0.9741,0.265040
1709,특명! 뭉이와 뭉글이를 보호하라!,"뭉이와 뭉글이의 운명을 상상해 보고 나니, 진짜로 내가 정성 들여서 만든 캐릭터들이...",0.9741,0.597572


In [68]:
# all_final_probs = []
# for i in range(len(output)):
#     if output["doc_prob"][i] 와 output["para_prob"][i]가 0.1슷하면
#         all_final_probs에 output["para_prob"] 추가
#     else 비슷하지않고 반대되는 값이라면 
        

공중 도덕의 의의와 필요성
공중 도덕의 의의와 필요성
공중 도덕의 의의와 필요성
공중 도덕의 의의와 필요성
풍습과 그 개선
풍습과 그 개선
풍습과 그 개선
생활의 안정과 가정
생활의 안정과 가정
생활의 안정과 가정
형제 자매와 친척
형제 자매와 친척
형제 자매와 친척
부부의 도리
부부의 도리
부부의 도리
옛날의 효도와 본 뜻
옛날의 효도와 본 뜻
옛날의 효도와 본 뜻
학교에서 지켜야할 도덕
학교에서 지켜야할 도덕
학교에서 지켜야할 도덕
생활과 근로정신
생활과 근로정신
생활과 근로정신
생활과 근로정신
공동생활과 자기 지위
공동생활과 자기 지위
공동생활과 자기 지위
노동의 신성
노동의 신성
노동의 신성
신용의 뜻과 성공의 기초
신용의 뜻과 성공의 기초
신용의 뜻과 성공의 기초
신용의 뜻과 성공의 기초
관용과 화합
관용과 화합
관용과 화합
관용과 화합
책임의 완수
책임의 완수
책임의 완수
책임의 완수
협동정신과 각 개인의 사회적 관련
협동정신과 각 개인의 사회적 관련
협동정신과 각 개인의 사회적 관련
법의 목적
법의 목적
야구의 성격
야구의 성격
야구의 성격
꽃을 나누어 보면
꽃을 나누어 보면
꽃을 나누어 보면
양심의 의의와 그 작용
양심의 의의와 그 작용
양심의 의의와 그 작용
인권의 본질과 사람의 권리
인권의 본질과 사람의 권리
인권의 본질과 사람의 권리
물건의 가치와 정신적 가치
물건의 가치와 정신적 가치
물건의 가치와 정신적 가치
개인의 도덕적 자각
개인의 도덕적 자각
개인의 도덕적 자각
개인의 도덕적 자각
인간 본성의 양면
인간 본성의 양면
인간 본성의 양면
두 가지 사상에 대한 비판
두 가지 사상에 대한 비판
참된 인생과 값이 있는 인생
참된 인생과 값이 있는 인생
참된 인생의 길을 찾아서
참된 인생의 길을 찾아서
참된 인생의 길을 찾아서
동기와 결과의 관계
동기와 결과의 관계
동기와 결과의 관계
인격의 본질
인격의 본질
인격의 본질
문장법과 문장도
문장법과 문장도
문장법과 문장도
문장법과 문장도
고운 음성과 바른 말
고운 음성과 바른 말
고운 음성과 바

In [ ]:
# for i in output:
# sample_submission = pd.read_csv('./data/sample_submission.csv', encoding='utf-8-sig')
# sample_submission['generated'] = all_test_preds

# sample_submission.to_csv(f'./att_submissionV5_doc_para.csv', index=False)



# print("✅ 예측 결과 저장 완료 → att_submissionV5_doc_para.csv")

In [61]:
output["para_prob"][7]

0.8177329

In [89]:
with torch.no_grad():
    i = 0
    for batch in tqdm(test_dataloader, desc="Predicting"):
        i+=1
        print("para : ", batch['paragraphs'])
        print("docs : ", batch['docs'])
        if i == 2:
            break

Predicting:   0% 1/1962 [00:00<00:01, 1231.45it/s]

para :  ['도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다. 그러므로 도덕은 어디까지나 정신의 문제이고, 각자의 마음씨에 달려있는 일이다. 여기에서 도덕의 문제는 철학적 이론으로 발전하였으며, 고상하고 심원한 이론 체계에 기울어지는 경향이 많았다. 이러한 경향으로 인해 도덕은 학식이 높은 특수한 사람만이 닦을 수 있는 것으로 여겨지며, 일반 사람은 도저히 지킬 수 없는 것처럼 오해받는 경우가 많았다. 그러나 도덕의 본질은 결코 이론에 있는 것이 아니라 실천에 있다. 이론이 필요하다면 그것은 오직 실천을 위한 도구로서만 의미를 가진다. 아무리 고상한 이론이라도 실천이 없다면 그 이론은 단순한 관념의 유희에 불과하다.']
docs :  ['도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다. 그러므로 도덕은 어디까지나 정신의 문제이고, 각자의 마음씨에 달려있는 일이다. 여기에서 도덕의 문제는 철학적 이론으로 발전하였으며, 고상하고 심원한 이론 체계에 기울어지는 경향이 많았다. 이러한 경향으로 인해 도덕은 학식이 높은 특수한 사람만이 닦을 수 있는 것으로 여겨지며, 일반 사람은 도저히 지킬 수 없는 것처럼 오해받는 경우가 많았다. 그러나 도덕의 본질은 결코 이론에 있는 것이 아니라 실천에 있다. 이론이 필요하다면 그것은 오직 실천을 위한 도구로서만 의미를 가진다. 아무리 고상한 이론이라도 실천이 없다면 그 이론은 단순한 관념의 유희에 불과하다. 도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없다. 개인적인 측면과 사회적인 측면은 서로 밀접하게 연결되어 있기 때문이다. 이는 인간이 본래 개인이면서 동시에 사회적 존재라는 사실에서 기인한다. 그러나 도덕의 문제를 다룰 때는 개인적인 측면을 강조하는 경우도 있고, 사회적인 측면을 강조하는 경우도 존재한다. 예를 들어, 인격의 자유나 양심의 가책, 선악의 판단과 같은 문제는 항상 개인이 중심이 되지만, 어른에 대한 예의, 길을 걷는 사람에 대한 친절, 빈곤한 사람에 대한 동